# DATASCI 151 — Quiz Guide
Dictionaries · pd.cut · groupby · agg · merge · query · chaining

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf  # only needed for linear regression (HW9)

## 1. Dictionaries

In [ ]:
# Create a dictionary: keys can be strings or ints, values can be anything
grades = {"Nancy": 92.8, "Bill": 87.3, "Boris": 82.3}

# Access by key
grades["Boris"]          # → 82.3

# Update a value
grades["Boris"] = 85

# Get all keys / values
grades.keys()
grades.values()

# Loop over key-value pairs
for key in grades.keys():
    print(key, grades[key])

In [ ]:
# Dictionary → DataFrame
# Keys become column names; values are lists (ALL lists must be the same length!)
state_dict = {
    "state_name":  ["Wisconsin", "Georgia", "Oregon"],
    "capital":     ["Madison",   "Atlanta",  "Salem"],
    "elec_votes":  [10, 16, 8]
}
df = pd.DataFrame(state_dict)
display(df)

## 2. Recoding Variables — pd.cut

In [ ]:
# pd.cut splits a numeric column into labeled bins
# IMPORTANT: you need N+1 bin edges for N labels
# right=True  → intervals are (left, right]  i.e. right endpoint is INCLUDED (default)
# right=False → intervals are [left, right)  i.e. left endpoint is INCLUDED

# --- Practice Quiz Q1: year → time period ---
year_bins   = [1940, 1967, 2003, np.inf]         # 4 edges → 3 intervals
year_labels = ["1941-1967", "1968-2003", "2004-onwards"]

df["time_period"] = pd.cut(x=df["year"],
                            bins=year_bins,
                            labels=year_labels)
                            # right=True by default → (1940,1967], (1967,2003], (2003,inf]

# Alt with right=False (left endpoint included):
year_bins2 = [1941, 1968, 2004, np.inf]
df["time_period"] = pd.cut(x=df["year"], bins=year_bins2,
                            labels=year_labels, right=False)
                            # → [1941,1968), [1968,2004), [2004,inf)

In [ ]:
# --- HW9 Q-b: wage → Low / Medium / High / Elite ---
bins   = [-np.inf, 50000, 100000, 200000, np.inf]  # always use -inf/inf to capture everything
labels = ["Low", "Medium", "High", "Elite"]

df["wage_level"] = pd.cut(df["wage_eur"],
                           bins=bins,
                           right=True,   # <=50k is Low, 50k< x <=100k is Medium, etc.
                           labels=labels)

# NOTE: values outside all bins get NaN — use -np.inf / np.inf to avoid this

## 3. Groupby

In [ ]:
# .groupby() returns a DataFrameGroupBy object — it's lazy (does nothing until you ask)
# The grouped column becomes the INDEX of the result, not a regular column

grp = df_results.groupby(by="driverId")

# Single stat on one column
grp["points"].mean()       # Series: mean points per driver
grp["points"].sum()
grp["points"].max()

# Multiple columns at once
grp[["points", "laps"]].mean()

# Group by multiple keys (pass a list)
df_results.groupby(["raceId", "constructorId"])["points"].mean()

## 4. Aggregation — .agg()

In [ ]:
# On a Series — pass a list of stat name strings
df["points"].agg(["mean", "std", "min", "max"])

# On a DataFrame — named output columns using tuple syntax
# new_col_name = ("source_column", "stat")
df.agg(
    mean_pts = ("points", "mean"),
    max_pts  = ("points", "max"),
    n_unique = ("driverId", "nunique"),
)

# Built-in stat strings: "mean" "sum" "min" "max" "std" "count" "nunique"
# Use len (no quotes) to count all rows including duplicates

In [ ]:
# groupby + agg  ← the main pattern
drivers_agg = (df_results
               .groupby("driverId")
               .agg(
                   avg_pts     = ("points", "mean"),
                   sd_pts      = ("points", "std"),
                   min_pts     = ("points", "min"),
                   max_pts     = ("points", "max"),
                   appearances = ("points", len),   # len = count rows, NO quotes
               ))

In [ ]:
# Custom function in .agg  (Practice Quiz Q6, HW9 Q-e)
# Write a function that takes a Series, returns a single number

def podium_finishes(col):
    return (col <= 3).sum()   # count entries <= 3

# Explicit loop version (same result):
def podium_finishes(col):
    count = 0
    for val in col:
        if val <= 3:
            count += 1
    return count

# Pass the function NAME — no quotes!
df_results.groupby("driverId").agg(
    avg_pts   = ("points",        "mean"),
    podiums   = ("positionOrder", podium_finishes),  # ← no quotes
)

## 5. Chaining — .query() → .groupby() → .agg() → .sort_values() → .iloc[]

In [ ]:
# Wrap in () to break across lines cleanly
# Can also use \ at end of line instead

# --- Practice Quiz Q2: group by country, sort by avg imdb descending ---
(df.groupby("country")
   .agg(avg_run  = ("runtime(minutes)", "mean"),
        max_run  = ("runtime(minutes)", "max"),
        avg_imdb = ("imdb_score",       "mean"),
        min_imdb = ("imdb_score",       "min"))
   .sort_values(by="avg_imdb", ascending=False))

In [ ]:
# --- Practice Quiz Q3: filter France, sort by year, get 2 oldest ---
df.query("country == 'France'").sort_values(by="year").iloc[:2, :]

In [ ]:
# --- Practice Quiz Q4: filter, groupby, agg, then filter again on result ---
(df_sprint.query("constructorId == 51")
          .groupby("driverId")
          .agg(max_laps = ("laps", "max"),
               avg_laps = ("laps", "mean"))
          .query("max_laps >= 20"))   # .query() works on aggregated df too

In [ ]:
# --- Practice Quiz Q5: new column, groupby two keys, sort, top 5 ---
df_results["race_duration_minutes"] = df_results["milliseconds"] / 6e4

(df_results
 .groupby(["raceId", "constructorId"])["race_duration_minutes"]
 .mean()
 .sort_values(ascending=False)
 .head(5))   # .head(5) same as .iloc[:5]

## 6. Merge — pd.merge()

In [ ]:
# Typical pattern: compute agg stats, then merge back into original df
df_agg = df_results.groupby("driverId").agg(
    avg_pts = ("points", "mean"),
    podiums = ("positionOrder", podium_finishes)
)

result = pd.merge(left  = df_results,
                  right = df_agg,
                  on    = "driverId",  # column that appears in both DataFrames
                  how   = "left")      # keep all rows from left df

# Multi-key merge (when you grouped by two columns)
result = pd.merge(df_results, teamrace_agg,
                  on=["raceId", "constructorId"],
                  how="left")

## 7. Sorting & Slicing Quick Reference

In [ ]:
# Sort
df.sort_values(by="col", ascending=False)  # descending
df.sort_values(by="col", ascending=True)   # ascending (default)

# iloc — position-based (integers only)
df.iloc[:5]       # first 5 rows
df.iloc[:5, :]    # first 5 rows, all columns
df.iloc[0, 2]     # row 0, column 2

# loc — label-based (index name or column name)
df.loc["mean"]           # row where index label == "mean"
df.loc[1, "points"]      # row index 1, column "points"

# head / tail
df.head(5)   # same as df.iloc[:5]
df.tail(5)

# Unique counts
df["col"].nunique()   # count of unique values
df["col"].unique()    # array of unique values
len(pd.unique(df["col"]))  # same as nunique

## 8. Common Mistakes to Avoid

In [ ]:
# 1. pd.cut: need N+1 edges for N labels
#    BAD:  bins=[0, 50, 100],  labels=["Low", "Med", "High"]  ← 3 labels, only 2 intervals
#    GOOD: bins=[0, 50, 100],  labels=["Low", "Med"]          ← 2 labels, 2 intervals

# 2. Custom function in .agg — NO quotes
#    BAD:  .agg(x=("col", "my_func"))   ← only works for built-ins
#    GOOD: .agg(x=("col",  my_func))    ← pass the function object

# 3. After groupby().agg(), the grouped column is the INDEX not a regular column
#    To turn it back into a column: df_agg.reset_index()

# 4. Values outside pd.cut bins → NaN (use -np.inf and np.inf to catch everything)

# 5. .query() can be used BEFORE groupby (filter rows) AND AFTER agg (filter groups)
#    df.query("x > 0").groupby("id").agg(...).query("avg > 5")